# Singapore Tuition Signup Seasonality Prediction Pipeline

Predicts when tuition demand peaks throughout the year so businesses know when to push advertisements and promotions.

**Pipeline overview:**
1. Collect external signals: MOE school calendar, SEAB exam dates, Google Trends, SingStat enrollment
2. Use Google Trends search volume as a demand proxy (proxy for signup intent)
3. Engineer calendar/event features
4. Train a Prophet model with Singapore-specific holidays and events
5. Forecast demand and identify optimal promotion windows

In [ ]:
%pip install prophet pytrends requests pandas numpy matplotlib seaborn scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
import warnings
from datetime import date, timedelta
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from pytrends.request import TrendReq
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_theme(style='whitegrid')

print('All imports successful')

## Section 1: MOE School Calendar & SEAB Exam Dates

Hardcoded from MOE press releases (https://www.moe.gov.sg/news/press-releases) and SEAB timetables.
Covers 2020-2026. Update annually when MOE publishes the following year's calendar (usually August).

In [ ]:
# MOE school term start/end dates and holiday periods by year
# Source: https://www.moe.gov.sg/news/press-releases (search 'school terms')
moe_calendar = {
    2020: {
        'term1': ('2020-01-02', '2020-03-13'), 'term2': ('2020-03-23', '2020-05-29'),
        'term3': ('2020-06-29', '2020-09-04'), 'term4': ('2020-09-14', '2020-11-20'),
        'march_hols': ('2020-03-14', '2020-03-22'), 'june_hols': ('2020-05-30', '2020-06-28'),
        'sept_hols': ('2020-09-05', '2020-09-13'), 'year_end_hols': ('2020-11-21', '2020-12-31'),
    },
    2021: {
        'term1': ('2021-01-04', '2021-03-12'), 'term2': ('2021-03-22', '2021-05-28'),
        'term3': ('2021-06-28', '2021-09-03'), 'term4': ('2021-09-13', '2021-11-19'),
        'march_hols': ('2021-03-13', '2021-03-21'), 'june_hols': ('2021-05-29', '2021-06-27'),
        'sept_hols': ('2021-09-04', '2021-09-12'), 'year_end_hols': ('2021-11-20', '2021-12-31'),
    },
    2022: {
        'term1': ('2022-01-03', '2022-03-11'), 'term2': ('2022-03-21', '2022-05-27'),
        'term3': ('2022-06-27', '2022-09-02'), 'term4': ('2022-09-12', '2022-11-18'),
        'march_hols': ('2022-03-12', '2022-03-20'), 'june_hols': ('2022-05-28', '2022-06-26'),
        'sept_hols': ('2022-09-03', '2022-09-11'), 'year_end_hols': ('2022-11-19', '2022-12-31'),
    },
    2023: {
        'term1': ('2023-01-03', '2023-03-10'), 'term2': ('2023-03-20', '2023-05-26'),
        'term3': ('2023-06-26', '2023-09-01'), 'term4': ('2023-09-11', '2023-11-17'),
        'march_hols': ('2023-03-11', '2023-03-19'), 'june_hols': ('2023-05-27', '2023-06-25'),
        'sept_hols': ('2023-09-02', '2023-09-10'), 'year_end_hols': ('2023-11-18', '2023-12-31'),
    },
    2024: {
        'term1': ('2024-01-02', '2024-03-08'), 'term2': ('2024-03-18', '2024-05-24'),
        'term3': ('2024-06-24', '2024-08-30'), 'term4': ('2024-09-09', '2024-11-15'),
        'march_hols': ('2024-03-09', '2024-03-17'), 'june_hols': ('2024-05-25', '2024-06-23'),
        'sept_hols': ('2024-08-31', '2024-09-08'), 'year_end_hols': ('2024-11-16', '2024-12-31'),
    },
    2025: {
        'term1': ('2025-01-02', '2025-03-14'), 'term2': ('2025-03-24', '2025-05-30'),
        'term3': ('2025-06-30', '2025-09-05'), 'term4': ('2025-09-15', '2025-11-21'),
        'march_hols': ('2025-03-15', '2025-03-23'), 'june_hols': ('2025-05-31', '2025-06-29'),
        'sept_hols': ('2025-09-06', '2025-09-14'), 'year_end_hols': ('2025-11-22', '2025-12-31'),
    },
    2026: {
        'term1': ('2026-01-02', '2026-03-13'), 'term2': ('2026-03-23', '2026-05-29'),
        'term3': ('2026-06-29', '2026-09-04'), 'term4': ('2026-09-14', '2026-11-20'),
        'march_hols': ('2026-03-14', '2026-03-22'), 'june_hols': ('2026-05-30', '2026-06-28'),
        'sept_hols': ('2026-09-05', '2026-09-13'), 'year_end_hols': ('2026-11-21', '2026-12-31'),
    },
}

# SEAB national exam windows and results release dates
# Source: https://www.seab.gov.sg/important-dates-for-candidates
seab_events = [
    # SA1 (school-based, approximate window)
    {'event': 'SA1_exam',      'year': 2020, 'start': '2020-04-27', 'end': '2020-05-08'},
    {'event': 'SA1_exam',      'year': 2021, 'start': '2021-04-26', 'end': '2021-05-07'},
    {'event': 'SA1_exam',      'year': 2022, 'start': '2022-04-25', 'end': '2022-05-06'},
    {'event': 'SA1_exam',      'year': 2023, 'start': '2023-04-24', 'end': '2023-05-05'},
    {'event': 'SA1_exam',      'year': 2024, 'start': '2024-04-22', 'end': '2024-05-03'},
    {'event': 'SA1_exam',      'year': 2025, 'start': '2025-04-28', 'end': '2025-05-09'},
    {'event': 'SA1_exam',      'year': 2026, 'start': '2026-04-27', 'end': '2026-05-08'},
    # SA2 (school-based)
    {'event': 'SA2_exam',      'year': 2020, 'start': '2020-09-28', 'end': '2020-10-16'},
    {'event': 'SA2_exam',      'year': 2021, 'start': '2021-09-27', 'end': '2021-10-15'},
    {'event': 'SA2_exam',      'year': 2022, 'start': '2022-09-26', 'end': '2022-10-14'},
    {'event': 'SA2_exam',      'year': 2023, 'start': '2023-09-25', 'end': '2023-10-13'},
    {'event': 'SA2_exam',      'year': 2024, 'start': '2024-09-23', 'end': '2024-10-11'},
    {'event': 'SA2_exam',      'year': 2025, 'start': '2025-09-29', 'end': '2025-10-17'},
    {'event': 'SA2_exam',      'year': 2026, 'start': '2026-09-28', 'end': '2026-10-16'},
    # PSLE (written papers)
    {'event': 'PSLE_exam',     'year': 2020, 'start': '2020-08-31', 'end': '2020-09-25'},
    {'event': 'PSLE_exam',     'year': 2021, 'start': '2021-08-30', 'end': '2021-09-24'},
    {'event': 'PSLE_exam',     'year': 2022, 'start': '2022-09-01', 'end': '2022-09-29'},
    {'event': 'PSLE_exam',     'year': 2023, 'start': '2023-08-31', 'end': '2023-09-28'},
    {'event': 'PSLE_exam',     'year': 2024, 'start': '2024-08-27', 'end': '2024-09-27'},
    {'event': 'PSLE_exam',     'year': 2025, 'start': '2025-08-25', 'end': '2025-09-26'},
    {'event': 'PSLE_exam',     'year': 2026, 'start': '2026-08-12', 'end': '2026-09-30'},
    # PSLE results
    {'event': 'PSLE_results',  'year': 2020, 'start': '2020-11-25', 'end': '2020-11-25'},
    {'event': 'PSLE_results',  'year': 2021, 'start': '2021-11-24', 'end': '2021-11-24'},
    {'event': 'PSLE_results',  'year': 2022, 'start': '2022-11-23', 'end': '2022-11-23'},
    {'event': 'PSLE_results',  'year': 2023, 'start': '2023-11-22', 'end': '2023-11-22'},
    {'event': 'PSLE_results',  'year': 2024, 'start': '2024-11-27', 'end': '2024-11-27'},
    {'event': 'PSLE_results',  'year': 2025, 'start': '2025-11-26', 'end': '2025-11-26'},
    {'event': 'PSLE_results',  'year': 2026, 'start': '2026-11-24', 'end': '2026-11-25'},
    # O-Level
    {'event': 'OLevel_exam',   'year': 2020, 'start': '2020-10-05', 'end': '2020-11-06'},
    {'event': 'OLevel_exam',   'year': 2021, 'start': '2021-10-04', 'end': '2021-11-05'},
    {'event': 'OLevel_exam',   'year': 2022, 'start': '2022-10-03', 'end': '2022-11-04'},
    {'event': 'OLevel_exam',   'year': 2023, 'start': '2023-10-02', 'end': '2023-11-03'},
    {'event': 'OLevel_exam',   'year': 2024, 'start': '2024-10-07', 'end': '2024-11-08'},
    {'event': 'OLevel_exam',   'year': 2025, 'start': '2025-10-06', 'end': '2025-11-07'},
    {'event': 'OLevel_exam',   'year': 2026, 'start': '2026-10-05', 'end': '2026-11-06'},
    # O-Level results (January)
    {'event': 'OLevel_results','year': 2021, 'start': '2021-01-11', 'end': '2021-01-11'},
    {'event': 'OLevel_results','year': 2022, 'start': '2022-01-12', 'end': '2022-01-12'},
    {'event': 'OLevel_results','year': 2023, 'start': '2023-01-11', 'end': '2023-01-11'},
    {'event': 'OLevel_results','year': 2024, 'start': '2024-01-10', 'end': '2024-01-10'},
    {'event': 'OLevel_results','year': 2025, 'start': '2025-01-14', 'end': '2025-01-14'},
    {'event': 'OLevel_results','year': 2026, 'start': '2026-01-13', 'end': '2026-01-15'},
    # A-Level
    {'event': 'ALevel_exam',   'year': 2020, 'start': '2020-10-08', 'end': '2020-11-20'},
    {'event': 'ALevel_exam',   'year': 2021, 'start': '2021-10-07', 'end': '2021-11-19'},
    {'event': 'ALevel_exam',   'year': 2022, 'start': '2022-10-06', 'end': '2022-11-18'},
    {'event': 'ALevel_exam',   'year': 2023, 'start': '2023-10-05', 'end': '2023-11-17'},
    {'event': 'ALevel_exam',   'year': 2024, 'start': '2024-10-10', 'end': '2024-11-22'},
    {'event': 'ALevel_exam',   'year': 2025, 'start': '2025-10-09', 'end': '2025-11-21'},
    {'event': 'ALevel_exam',   'year': 2026, 'start': '2026-10-08', 'end': '2026-11-27'},
    # A-Level results (February)
    {'event': 'ALevel_results','year': 2021, 'start': '2021-02-26', 'end': '2021-02-26'},
    {'event': 'ALevel_results','year': 2022, 'start': '2022-02-25', 'end': '2022-02-25'},
    {'event': 'ALevel_results','year': 2023, 'start': '2023-02-24', 'end': '2023-02-24'},
    {'event': 'ALevel_results','year': 2024, 'start': '2024-02-22', 'end': '2024-02-22'},
    {'event': 'ALevel_results','year': 2025, 'start': '2025-02-21', 'end': '2025-02-21'},
    {'event': 'ALevel_results','year': 2026, 'start': '2026-02-19', 'end': '2026-02-23'},
]

df_events = pd.DataFrame(seab_events)
df_events['start'] = pd.to_datetime(df_events['start'])
df_events['end']   = pd.to_datetime(df_events['end'])
print(f'Loaded {len(df_events)} exam/results events')
display(spark.createDataFrame(df_events))

## Section 2: Google Trends Data

Pulls weekly search volume for tuition-related queries in Singapore via `pytrends`.
This is a strong leading indicator — search interest typically precedes signups by 2-4 weeks.

> **Note:** pytrends is an unofficial API wrapper and may hit rate limits. If it fails, the cell catches the error and continues with a warning.

In [ ]:
TRENDS_KEYWORDS = [
    'tuition singapore',
    'math tuition',
    'PSLE tuition',
    'english tuition',
    'science tuition',
]

def fetch_trends(keywords, geo='SG', timeframe='2020-01-01 2026-04-30'):
    pytrends = TrendReq(hl='en-US', tz=480)  # SGT = UTC+8
    frames = []
    # pytrends allows max 5 keywords per request
    for i in range(0, len(keywords), 5):
        batch = keywords[i:i+5]
        pytrends.build_payload(batch, geo=geo, timeframe=timeframe)
        df = pytrends.interest_over_time()
        if not df.empty:
            df = df.drop(columns=['isPartial'], errors='ignore')
            frames.append(df)
    return pd.concat(frames, axis=1) if frames else pd.DataFrame()

try:
    df_trends = fetch_trends(TRENDS_KEYWORDS)
    df_trends.index.name = 'date'
    df_trends = df_trends.reset_index()
    df_trends['date'] = pd.to_datetime(df_trends['date'])
    # Composite score: mean across all keywords, normalised 0-100
    kw_cols = [c for c in df_trends.columns if c != 'date']
    df_trends['trends_composite'] = df_trends[kw_cols].mean(axis=1)
    print(f'Google Trends data loaded: {len(df_trends)} weekly rows, {df_trends.date.min()} to {df_trends.date.max()}')
    display(spark.createDataFrame(df_trends[['date','trends_composite'] + kw_cols]))
except Exception as e:
    print(f'WARNING: Google Trends fetch failed ({e}). Continuing without trends data.')
    df_trends = pd.DataFrame(columns=['date','trends_composite'])

## Section 3: SingStat Enrollment Data

Annual student enrollment by school type from data.gov.sg (dataset `d_e24eb88daad6743854009265c61ee0d8`).
Used as a slow-moving baseline — larger cohorts = larger addressable market.

In [ ]:
DATAGOV_DATASET_ID = 'd_e24eb88daad6743854009265c61ee0d8'
DATAGOV_API_URL = f'https://data.gov.sg/api/action/datastore_search?resource_id={DATAGOV_DATASET_ID}&limit=500'

try:
    resp = requests.get(DATAGOV_API_URL, timeout=15)
    resp.raise_for_status()
    records = resp.json()['result']['records']
    df_enrollment = pd.DataFrame(records)
    # Clean numeric columns
    for col in df_enrollment.columns:
        if col not in ('_id', 'level_of_education', 'type_of_school'):
            df_enrollment[col] = pd.to_numeric(df_enrollment[col], errors='coerce')
    print(f'Enrollment data: {len(df_enrollment)} rows, columns: {list(df_enrollment.columns)}')
    display(spark.createDataFrame(df_enrollment.astype(str)))
except Exception as e:
    print(f'WARNING: data.gov.sg fetch failed ({e}). Using fallback enrollment estimates.')
    # Approximate annual primary+secondary enrollment 2015-2025 (SingStat published figures)
    df_enrollment = pd.DataFrame({
        'year': list(range(2015, 2026)),
        'primary_enrollment': [260000,258000,255000,252000,249000,247000,245000,244000,243000,242000,241000],
        'secondary_enrollment': [198000,197000,196000,195000,194000,193000,191000,189000,187000,185000,183000],
    })
    df_enrollment['total_enrollment'] = df_enrollment['primary_enrollment'] + df_enrollment['secondary_enrollment']
    print('Using fallback enrollment estimates.')
    display(spark.createDataFrame(df_enrollment))

## Section 4: Demand Data — Google Trends as Signup Proxy

Since internal signup records are not available, we use **Google Trends search volume** as a demand proxy.
Search interest for tuition-related queries is a well-accepted proxy for consumer intent and is used in
academic nowcasting research. The composite score (mean across all tracked keywords) becomes our target variable `y`.

> Data flows from Section 2. If the Trends fetch failed there, re-run that cell first.

In [ ]:
# Use Google Trends composite score as the demand proxy target
# Search volume for tuition queries is a validated proxy for consumer intent
if df_trends.empty or "trends_composite" not in df_trends.columns:
    raise RuntimeError("Google Trends data is missing — re-run Section 2 before continuing.")

df_signups = df_trends[["date", "trends_composite"]].copy()
df_signups = df_signups.rename(columns={"trends_composite": "signups"})
df_signups["date"] = pd.to_datetime(df_signups["date"])

# Trends is weekly — resample to daily via forward fill
df_signups = (
    df_signups
    .set_index("date")
    .resample("D")
    .ffill()
    .reset_index()
)

print(f"Demand proxy: {len(df_signups)} daily rows, {df_signups.date.min().date()} to {df_signups.date.max().date()}")
print(df_signups.describe())
display(spark.createDataFrame(df_signups))

## Section 5: Feature Engineering

Build a daily feature table combining all data sources into one model-ready DataFrame.

In [ ]:
df = df_signups.copy()
df['date'] = pd.to_datetime(df['date'])

# --- Calendar features ---
df['year']       = df['date'].dt.year
df['month']      = df['date'].dt.month
df['week']       = df['date'].dt.isocalendar().week.astype(int)
df['dayofweek']  = df['date'].dt.dayofweek  # 0=Mon
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

# --- School holiday flag ---
def is_school_holiday(d):
    cal = moe_calendar.get(d.year, {})
    for period in ['march_hols', 'june_hols', 'sept_hols', 'year_end_hols']:
        if period in cal:
            s, e = pd.to_datetime(cal[period][0]), pd.to_datetime(cal[period][1])
            if s <= d <= e:
                return 1
    return 0

df['is_school_holiday'] = df['date'].apply(is_school_holiday)

# --- Which term we are in (1-4, 0 = holiday) ---
def get_term(d):
    cal = moe_calendar.get(d.year, {})
    for i, term in enumerate(['term1','term2','term3','term4'], 1):
        if term in cal:
            s, e = pd.to_datetime(cal[term][0]), pd.to_datetime(cal[term][1])
            if s <= d <= e:
                return i
    return 0

df['school_term'] = df['date'].apply(get_term)

# --- Days until next exam / days since last results ---
exam_events    = df_events[df_events['event'].str.contains('exam')].copy()
results_events = df_events[df_events['event'].str.contains('results')].copy()

def days_until_next_exam(d):
    future = exam_events[exam_events['start'] >= d]
    if future.empty:
        return 365
    return (future['start'].min() - d).days

def days_since_last_results(d):
    past = results_events[results_events['start'] <= d]
    if past.empty:
        return 365
    return (d - past['start'].max()).days

df['days_to_exam']       = df['date'].apply(days_until_next_exam)
df['days_since_results'] = df['date'].apply(days_since_last_results)

# Urgency curves: exponential decay (peaks as exam approaches / just after results)
df['exam_urgency']     = np.exp(-df['days_to_exam'] / 14).clip(0, 1)
df['results_urgency']  = np.exp(-df['days_since_results'] / 14).clip(0, 1)

# --- Merge Google Trends (weekly -> daily via ffill) ---
if not df_trends.empty and 'trends_composite' in df_trends.columns:
    trends_daily = df_trends[['date','trends_composite']].set_index('date').resample('D').ffill().reset_index()
    df = df.merge(trends_daily, on='date', how='left')
    df['trends_composite'] = df['trends_composite'].fillna(df['trends_composite'].median())
else:
    df['trends_composite'] = 50.0  # neutral placeholder

# --- Merge enrollment (annual -> daily via ffill) ---
if 'year' in df_enrollment.columns:
    enroll_map = df_enrollment.set_index('year')['total_enrollment'].to_dict() \
        if 'total_enrollment' in df_enrollment.columns \
        else {y: 430000 for y in range(2015, 2027)}
    df['total_enrollment'] = df['year'].map(enroll_map).fillna(430000)
else:
    df['total_enrollment'] = 430000

print(f'Feature table: {df.shape[0]} rows x {df.shape[1]} columns')
display(spark.createDataFrame(df.head(30).astype(str)))

## Section 6: Exploratory Data Analysis

In [ ]:
# Raw time series
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Daily signups with 30-day rolling average
df_plot = df.copy()
df_plot['rolling30'] = df_plot['signups'].rolling(30).mean()
axes[0].plot(df_plot['date'], df_plot['signups'], alpha=0.3, color='steelblue', lw=0.8)
axes[0].plot(df_plot['date'], df_plot['rolling30'], color='steelblue', lw=2, label='30-day avg')
for _, ev in df_events[df_events['event'].str.contains('results')].iterrows():
    axes[0].axvline(ev['start'], color='red', alpha=0.4, lw=1, linestyle='--')
axes[0].set_title('Daily Signups (red dashed = results release dates)')
axes[0].set_ylabel('Signups')
axes[0].legend()

# Average signups by month (aggregated across all years)
monthly_avg = df.groupby('month')['signups'].mean()
axes[1].bar(monthly_avg.index, monthly_avg.values, color='steelblue')
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
axes[1].set_title('Average Daily Signups by Month')
axes[1].set_ylabel('Avg Signups')

# Exam urgency vs signups
axes[2].scatter(df['exam_urgency'], df['signups'], alpha=0.1, s=5, color='steelblue')
axes[2].set_xlabel('Exam Urgency Score')
axes[2].set_ylabel('Signups')
axes[2].set_title('Exam Urgency vs Daily Signups')

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: avg signups by year x month
pivot = df.pivot_table(values='signups', index='year', columns='month', aggfunc='mean')
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
plt.figure(figsize=(14, 4))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5)
plt.title('Average Daily Signups — Year x Month Heatmap')
plt.tight_layout()
plt.show()

## Section 7: Prophet Model

Prophet handles:
- **Trend**: long-run growth in signups
- **Weekly seasonality**: weekday vs weekend patterns
- **Yearly seasonality**: the Singapore exam calendar cycle
- **Custom holidays/events**: results releases and exam windows added as named regressors

Additional external regressors (exam urgency, results urgency, Google Trends) are added via `add_regressor()`.

In [ ]:
# Build Prophet holidays dataframe (named events with windows)
def events_to_prophet_holidays(df_ev, lower_window=-3, upper_window=14):
    rows = []
    for _, row in df_ev.iterrows():
        d = row['start']
        while d <= row['end']:
            rows.append({'holiday': row['event'], 'ds': d,
                         'lower_window': lower_window, 'upper_window': upper_window})
            d += timedelta(days=1)
    return pd.DataFrame(rows)

sg_holidays = events_to_prophet_holidays(df_events)

# Add school holidays as a separate holiday event
school_hol_rows = []
for d in df[df['is_school_holiday'] == 1]['date']:
    school_hol_rows.append({'holiday': 'school_holiday', 'ds': d, 'lower_window': 0, 'upper_window': 0})
sg_holidays = pd.concat([sg_holidays, pd.DataFrame(school_hol_rows)], ignore_index=True)

print(f'Prophet holidays table: {len(sg_holidays)} rows, {sg_holidays.holiday.nunique()} distinct events')

In [ ]:
# Prepare training dataframe in Prophet format (ds, y)
prophet_df = df[['date','signups','exam_urgency','results_urgency','trends_composite']].copy()
prophet_df = prophet_df.rename(columns={'date': 'ds', 'signups': 'y'})

# Normalise regressors to [0,1]
for col in ['exam_urgency','results_urgency','trends_composite']:
    mn, mx = prophet_df[col].min(), prophet_df[col].max()
    prophet_df[col] = (prophet_df[col] - mn) / (mx - mn + 1e-9)

# Train/test split: hold out last 90 days
cutoff = prophet_df['ds'].max() - pd.Timedelta(days=90)
train_df = prophet_df[prophet_df['ds'] <= cutoff]
test_df  = prophet_df[prophet_df['ds'] >  cutoff]
print(f'Train: {len(train_df)} rows up to {cutoff.date()},  Test: {len(test_df)} rows')

# Instantiate and train Prophet
m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    holidays=sg_holidays,
    seasonality_mode='multiplicative',  # better for series with growing variance
    changepoint_prior_scale=0.1,
    holidays_prior_scale=10.0,
)
m.add_regressor('exam_urgency',    standardize=False)
m.add_regressor('results_urgency', standardize=False)
m.add_regressor('trends_composite', standardize=False)

m.fit(train_df)
print('Model trained.')

In [ ]:
# --- In-sample + test forecast ---
forecast_train = m.predict(train_df)
forecast_test  = m.predict(test_df)

fig = m.plot(pd.concat([forecast_train, forecast_test]))
plt.title('Prophet Forecast vs Actuals')
plt.tight_layout()
plt.show()

fig2 = m.plot_components(pd.concat([forecast_train, forecast_test]))
plt.tight_layout()
plt.show()

# Test MAE / MAPE
merged = test_df.merge(forecast_test[['ds','yhat']], on='ds')
mae  = (merged['y'] - merged['yhat']).abs().mean()
mape = ((merged['y'] - merged['yhat']).abs() / merged['y'].replace(0, np.nan)).mean() * 100
print(f'Test MAE: {mae:.1f} signups/day   |   Test MAPE: {mape:.1f}%')

## Section 8: Cross-Validation

Rolling-window cross-validation: trains on 18 months, evaluates on the following 3 months, steps forward every month.

In [ ]:
df_cv = cross_validation(
    m,
    initial='548 days',   # ~18 months initial training
    period='30 days',     # step forward 1 month
    horizon='90 days',    # evaluate 3 months ahead
    parallel='threads',
)
df_perf = performance_metrics(df_cv)
print(df_perf[['horizon','mape','rmse','mae']].to_string(index=False))

fig = plot_cross_validation_metric(df_cv, metric='mape')
plt.title('MAPE by Forecast Horizon (Cross-Validation)')
plt.tight_layout()
plt.show()

## Section 9: 12-Month Forecast & Optimal Promotion Windows

Forecast the next 12 months and highlight the highest-demand windows — these are the best times to push ads.
Low-demand periods are when promotions/discounts are most needed to drive signups.

In [ ]:
FORECAST_DAYS = 365
last_date = prophet_df['ds'].max()

future = m.make_future_dataframe(periods=FORECAST_DAYS, freq='D')

# Fill regressors for future dates
def future_exam_urgency(d):
    future_exams = exam_events[exam_events['start'] >= d]
    if future_exams.empty:
        return 0.0
    days = (future_exams['start'].min() - d).days
    return float(np.exp(-days / 14))

def future_results_urgency(d):
    past_results = results_events[results_events['start'] <= d]
    if past_results.empty:
        return 0.0
    days = (d - past_results['start'].max()).days
    return float(np.exp(-days / 14))

future['exam_urgency']     = future['ds'].apply(future_exam_urgency)
future['results_urgency']  = future['ds'].apply(future_results_urgency)
future['trends_composite'] = 0.5  # assume median trend if unknown

# Normalise to same scale as training
for col in ['exam_urgency','results_urgency']:
    mn = prophet_df[col].min()
    mx = prophet_df[col].max()
    future[col] = (future[col] - mn) / (mx - mn + 1e-9)

forecast = m.predict(future)

# Focus on the future horizon only
fcast_future = forecast[forecast['ds'] > last_date].copy()
fcast_future['yhat'] = fcast_future['yhat'].clip(0)

# Weekly aggregation for readability
fcast_weekly = fcast_future.set_index('ds')[['yhat','yhat_lower','yhat_upper']].resample('W').sum()

# Classify weeks into demand tiers
p33 = fcast_weekly['yhat'].quantile(0.33)
p67 = fcast_weekly['yhat'].quantile(0.67)
fcast_weekly['tier'] = pd.cut(
    fcast_weekly['yhat'],
    bins=[-np.inf, p33, p67, np.inf],
    labels=['LOW (run promos)', 'MEDIUM', 'HIGH (push ads)']
)

# Plot
fig, ax = plt.subplots(figsize=(16, 6))
colors = {'LOW (run promos)': '#d62728', 'MEDIUM': '#ff7f0e', 'HIGH (push ads)': '#2ca02c'}
for tier, color in colors.items():
    mask = fcast_weekly['tier'] == tier
    ax.bar(fcast_weekly.index[mask], fcast_weekly['yhat'][mask], width=6, color=color, alpha=0.8, label=tier)
ax.fill_between(fcast_weekly.index, fcast_weekly['yhat_lower'].clip(0), fcast_weekly['yhat_upper'],
                alpha=0.15, color='grey', label='80% interval')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
ax.set_title('12-Month Weekly Signup Forecast — Demand Tiers')
ax.set_ylabel('Projected Weekly Signups')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Actionable summary table
summary = fcast_weekly.reset_index()[['ds','yhat','tier']].copy()
summary.columns = ['week_start','projected_signups','action']
summary['projected_signups'] = summary['projected_signups'].round(0).astype(int)
summary['week_start'] = summary['week_start'].dt.date

print('=== RECOMMENDED AD/PROMO CALENDAR ===')
print()
print('HIGH demand weeks — maximise ad spend:')
high = summary[summary['action'] == 'HIGH (push ads)']
print(high.to_string(index=False))
print()
print('LOW demand weeks — run discount promotions to stimulate demand:')
low = summary[summary['action'] == 'LOW (run promos)']
print(low.to_string(index=False))

display(spark.createDataFrame(summary))

In [ ]:
# Save forecast to Delta table for downstream use
# Uncomment and update the catalog/schema to save

# forecast_spark = spark.createDataFrame(summary)
# forecast_spark.write.format('delta').mode('overwrite').saveAsTable('your_catalog.your_schema.tuition_forecast')
# print('Saved to Delta table.')

print('Pipeline complete.')